# Notebook 6: Feature Engineering

### Objective

The objective of this notebook is to transform the cleaned retail and Census datasets into meaningful analytical features for retail location intelligence.

Features related to demographic demand, workforce participation, literacy, household concentration, and competitive retail presence will be prepared for the location recommendation system.

The engineered dataset will later serve as the input for location scoring and advanced machine learning or deep learning analysis.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving retail_demand_index.csv to retail_demand_index.csv
Saving clean_darkstores.csv to clean_darkstores.csv


In [ ]:
stores = pd.read_csv("clean_darkstores.csv")
demand = pd.read_csv("retail_demand_index.csv")

print("Dark Store Dataset:", stores.shape)
print("Retail Demand Dataset:", demand.shape)

Dark Store Dataset: (4081, 9)
Retail Demand Dataset: (1920, 99)


In [ ]:
stores.head()

,id,accuracy,latitude,longitude,brand,name,city,state,locality
0,42985,43.0,8.478260,76.954711,Blinkit,NaN,NaN,NaN,NaN
1,40038,13.0,8.525199,76.955395,Blinkit,NaN,NaN,NaN,NaN
2,38514,30.0,9.587031,76.535782,Blinkit,NaN,NaN,NaN,NaN
3,38406,48.0,9.922391,78.095686,Blinkit,NaN,NaN,NaN,NaN
4,39056,0.0,9.949993,76.253442,Blinkit,NaN,NaN,NaN,NaN


In [ ]:
demand.head()

,State,District,Subdistt,Town/Village,Ward,EB,Level,Name,TRU,No_HH,...,MARG_OT_0_3_M,MARG_OT_0_3_F,NON_WORK_P,NON_WORK_M,NON_WORK_F,Literacy_Rate,Workforce_Rate,Average_Household_Size,Population_Rank,Retail_Demand_Index
0,1,1,0,0,0,0,DISTRICT,Kupwara,Total,113929,...,4727,2642,641290,283291,357999,50.51,26.32,7.64,983.0,0.273
1,1,1,0,0,0,0,DISTRICT,Kupwara,Rural,101930,...,4149,2495,569632,250046,319586,48.98,25.60,7.51,1045.0,0.250
2,1,1,0,0,0,0,DISTRICT,Kupwara,Urban,11999,...,578,147,71658,33245,38413,61.74,31.58,8.73,1683.0,0.195
3,1,2,0,0,0,0,DISTRICT,Badgam,Total,103363,...,2393,2281,538879,235463,303416,44.53,28.51,7.29,1057.0,0.243
4,1,2,0,0,0,0,DISTRICT,Badgam,Rural,89417,...,1740,1845,474065,208816,265249,42.43,27.72,7.33,1119.0,0.220


In [ ]:
# Check Dataset Columns

print("Dark Store Columns:")
print(stores.columns.tolist())

print("\nRetail Demand Columns:")
print(demand.columns.tolist())

Dark Store Columns:
['id', 'accuracy', 'latitude', 'longitude', 'brand', 'name', 'city', 'state', 'locality']

Retail Demand Columns:
['State', 'District', 'Subdistt', 'Town/Village', 'Ward', 'EB', 'Level', 'Name', 'TRU', 'No_HH', 'TOT_P', 'TOT_M', 'TOT_F', 'P_06', 'M_06', 'F_06', 'P_SC', 'M_SC', 'F_SC', 'P_ST', 'M_ST', 'F_ST', 'P_LIT', 'M_LIT', 'F_LIT', 'P_ILL', 'M_ILL', 'F_ILL', 'TOT_WORK_P', 'TOT_WORK_M', 'TOT_WORK_F', 'MAINWORK_P', 'MAINWORK_M', 'MAINWORK_F', 'MAIN_CL_P', 'MAIN_CL_M', 'MAIN_CL_F', 'MAIN_AL_P', 'MAIN_AL_M', 'MAIN_AL_F', 'MAIN_HH_P', 'MAIN_HH_M', 'MAIN_HH_F', 'MAIN_OT_P', 'MAIN_OT_M', 'MAIN_OT_F', 'MARGWORK_P', 'MARGWORK_M', 'MARGWORK_F', 'MARG_CL_P', 'MARG_CL_M', 'MARG_CL_F', 'MARG_AL_P', 'MARG_AL_M', 'MARG_AL_F', 'MARG_HH_P', 'MARG_HH_M', 'MARG_HH_F', 'MARG_OT_P', 'MARG_OT_M', 'MARG_OT_F', 'MARGWORK_3_6_P', 'MARGWORK_3_6_M', 'MARGWORK_3_6_F', 'MARG_CL_3_6_P', 'MARG_CL_3_6_M', 'MARG_CL_3_6_F', 'MARG_AL_3_6_P', 'MARG_AL_3_6_M', 'MARG_AL_3_6_F', 'MARG_HH_3_6_P', 'MARG

In [ ]:
# Create Geographic Location Groups

stores["location_lat"] = stores["latitude"].round(1)
stores["location_lon"] = stores["longitude"].round(1)

print(
    "Geographic Location Groups:",
    stores[["location_lat", "location_lon"]]
    .drop_duplicates()
    .shape[0]
)

Geographic Location Groups: 607


In [ ]:
# Create Location Competition Features

location_competition = (
    stores
    .groupby(["location_lat", "location_lon"])
    .agg(
        Total_Dark_Stores=("id", "count"),
        Competing_Brands=("brand", "nunique")
    )
    .reset_index()
)

location_competition.head()

,location_lat,location_lon,Total_Dark_Stores,Competing_Brands
0,8.2,77.4,1,1
1,8.5,76.9,3,1
2,8.5,77.0,8,2
3,8.6,76.9,2,1
4,8.7,76.8,1,1


In [ ]:
# Create Brand Presence Features

brand_presence = (
    stores
    .pivot_table(
        index=["location_lat", "location_lon"],
        columns="brand",
        values="id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

brand_presence.columns.name = None

brand_presence.columns = [
    str(col)
    .strip()
    .lower()
    .replace(" ", "_")
    for col in brand_presence.columns
]

brand_presence.head()

,location_lat,location_lon,blinkit,swiggy_instamart,zepto
0,8.2,77.4,0,1,0
1,8.5,76.9,0,3,0
2,8.5,77.0,2,6,0
3,8.6,76.9,0,2,0
4,8.7,76.8,0,1,0


In [ ]:
# Combine Location Competition Features

competition_features = location_competition.merge(
    brand_presence,
    on=["location_lat", "location_lon"],
    how="left"
)

competition_features["Competition_Intensity"] = (
    competition_features["Total_Dark_Stores"] *
    competition_features["Competing_Brands"]
)

competition_features.sort_values(
    "Competition_Intensity",
    ascending=False
).head(10)

,location_lat,location_lon,Total_Dark_Stores,Competing_Brands,blinkit,swiggy_instamart,zepto,Competition_Intensity
70,12.9,77.6,88,3,34,23,31,264
495,28.6,77.4,72,3,33,15,24,216
80,13.0,77.7,72,3,28,18,26,216
79,13.0,77.6,58,3,23,14,21,174
158,17.4,78.4,58,3,22,17,19,174
163,17.5,78.4,57,3,18,15,24,171
492,28.6,77.1,54,3,23,13,18,162
494,28.6,77.3,52,3,24,12,16,156
205,19.1,72.9,52,3,20,11,21,156
71,12.9,77.7,52,3,17,11,24,156


In [ ]:
# Add Competition Features to Stores

stores_features = stores.merge(
    competition_features,
    on=["location_lat", "location_lon"],
    how="left"
)

print("Stores:", len(stores_features))
print("Columns:", stores_features.shape[1])

stores_features.head()


Stores: 4081
Columns: 17


,id,accuracy,latitude,longitude,brand,name,city,state,locality,location_lat,location_lon,Total_Dark_Stores,Competing_Brands,blinkit,swiggy_instamart,zepto,Competition_Intensity
0,42985,43.0,8.478260,76.954711,Blinkit,NaN,NaN,NaN,NaN,8.5,77.0,8,2,2,6,0,16
1,40038,13.0,8.525199,76.955395,Blinkit,NaN,NaN,NaN,NaN,8.5,77.0,8,2,2,6,0,16
2,38514,30.0,9.587031,76.535782,Blinkit,NaN,NaN,NaN,NaN,9.6,76.5,2,2,1,1,0,4
3,38406,48.0,9.922391,78.095686,Blinkit,NaN,NaN,NaN,NaN,9.9,78.1,8,3,1,3,4,24
4,39056,0.0,9.949993,76.253442,Blinkit,NaN,NaN,NaN,NaN,9.9,76.3,4,3,1,2,1,12


In [ ]:
# Create Same Brand and Competitor Features

brand_column_map = {
    "Blinkit": "blinkit",
    "Swiggy Instamart": "swiggy_instamart",
    "Zepto": "zepto"
}

stores_features["Same_Brand_Stores"] = stores_features.apply(
    lambda row: row[brand_column_map[row["brand"]]],
    axis=1
)

stores_features["Competitor_Stores"] = (
    stores_features["Total_Dark_Stores"] -
    stores_features["Same_Brand_Stores"]
)

stores_features["Competitor_Ratio"] = (
    stores_features["Competitor_Stores"] /
    stores_features["Total_Dark_Stores"]
)

stores_features["Brand_Share"] = (
    stores_features["Same_Brand_Stores"] /
    stores_features["Total_Dark_Stores"]
)

print("Competition features created successfully.")

stores_features[
    [
        "brand",
        "Total_Dark_Stores",
        "Same_Brand_Stores",
        "Competitor_Stores",
        "Competitor_Ratio",
        "Brand_Share"
    ]
].head(10)

Competition features created successfully.


,brand,Total_Dark_Stores,Same_Brand_Stores,Competitor_Stores,Competitor_Ratio,Brand_Share
0,Blinkit,8,2,6,0.750000,0.250000
1,Blinkit,8,2,6,0.750000,0.250000
2,Blinkit,2,1,1,0.500000,0.500000
3,Blinkit,8,1,7,0.875000,0.125000
4,Blinkit,4,1,3,0.750000,0.250000
5,Blinkit,21,6,15,0.714286,0.285714
6,Blinkit,21,6,15,0.714286,0.285714
7,Blinkit,21,6,15,0.714286,0.285714
8,Blinkit,21,6,15,0.714286,0.285714
9,Blinkit,8,1,7,0.875000,0.125000


In [ ]:
# Summarize Competition Features

competition_columns = [
    "Total_Dark_Stores",
    "Competing_Brands",
    "Same_Brand_Stores",
    "Competitor_Stores",
    "Competitor_Ratio",
    "Brand_Share",
    "Competition_Intensity"
]

stores_features[
    competition_columns
].describe().round(2)

,Total_Dark_Stores,Competing_Brands,Same_Brand_Stores,Competitor_Stores,Competitor_Ratio,Brand_Share,Competition_Intensity
count,4081.00,4081.00,4081.00,4081.00,4081.00,4081.00,4081.00
mean,25.28,2.70,9.45,15.83,0.55,0.45,74.91
std,21.38,0.62,8.01,14.40,0.21,0.21,65.01
min,1.00,1.00,1.00,0.00,0.00,0.09,1.00
25%,8.00,3.00,3.00,4.00,0.50,0.31,18.00
50%,20.00,3.00,7.00,12.00,0.60,0.40,60.00
75%,41.00,3.00,14.00,25.00,0.69,0.50,123.00
max,88.00,3.00,34.00,65.00,0.91,1.00,264.00


In [ ]:
# Prepare District Demand Features

district_demand = demand[
    (demand["Level"] == "DISTRICT") &
    (demand["TRU"] == "Total")
].copy()

district_demand = district_demand[
    [
        "State",
        "Name",
        "TOT_P",
        "No_HH",
        "Literacy_Rate",
        "Workforce_Rate",
        "Average_Household_Size",
        "Retail_Demand_Index"
    ]
]

district_demand = district_demand.rename(
    columns={
        "Name": "District",
        "TOT_P": "Population",
        "No_HH": "Households"
    }
)

print("District Records:", len(district_demand))

district_demand.head()

District Records: 640


,State,District,Population,Households,Literacy_Rate,Workforce_Rate,Average_Household_Size,Retail_Demand_Index
0,1,Kupwara,870354,113929,50.51,26.32,7.64,0.273
3,1,Badgam,753745,103363,44.53,28.51,7.29,0.243
6,1,Leh(Ladakh),133487,21909,70.25,56.24,6.09,0.479
9,1,Kargil,140802,18338,61.25,36.84,7.68,0.282
12,1,Punch,476835,90261,54.89,33.85,5.28,0.287


In [ ]:
# Check Retail Demand Distribution

district_demand[
    [
        "Population",
        "Households",
        "Literacy_Rate",
        "Workforce_Rate",
        "Average_Household_Size",
        "Retail_Demand_Index"
    ]
].describe().round(2)

,Population,Households,Literacy_Rate,Workforce_Rate,Average_Household_Size,Retail_Demand_Index
count,640.00,640.00,640.00,640.00,640.00,640.00
mean,1891960.90,389846.35,62.46,41.20,4.97,0.58
std,1544380.29,337542.76,10.53,7.03,0.72,0.14
min,8004.00,1952.00,28.77,25.83,3.42,0.16
25%,817861.00,169401.75,55.13,35.58,4.46,0.48
50%,1557367.00,314171.50,62.03,41.20,4.92,0.57
75%,2583551.25,534074.75,70.49,46.58,5.33,0.68
max,11060148.00,2529165.00,88.74,66.90,7.68,0.90


In [ ]:
# Identify High Demand Districts

top_demand = district_demand.sort_values(
    "Retail_Demand_Index",
    ascending=False
)

top_demand[
    [
        "State",
        "District",
        "Population",
        "Households",
        "Literacy_Rate",
        "Workforce_Rate",
        "Retail_Demand_Index"
    ]
].head(10)

,State,District,Population,Households,Literacy_Rate,Workforce_Rate,Retail_Demand_Index
1713,29,Bangalore,9621551,2393845,78.08,44.14,0.899
1560,27,Pune,9429408,2151503,76.06,42.94,0.878
1551,27,Mumbai Suburban,9356962,2105604,80.96,39.92,0.877
1806,33,Chennai,4646732,1154982,81.27,39.11,0.864
1893,33,Coimbatore,3458045,958035,76.23,45.34,0.862
1023,19,Kolkata,4496694,1024928,79.80,39.93,0.861
1512,27,Nagpur,4653570,1041544,78.95,40.15,0.860
1473,24,Surat,6081322,1333200,75.17,41.99,0.860
1554,27,Mumbai,3085411,674339,81.32,41.63,0.851
1563,27,Ahmadnagar,4543159,930024,69.38,48.53,0.851


In [ ]:
# Create Demand Categories

district_demand["Demand_Category"] = pd.qcut(
    district_demand["Retail_Demand_Index"],
    q=4,
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ]
)

district_demand["Demand_Category"].value_counts()

,count
Demand_Category,
Moderate,163
Low,160
Very High,159
High,158


In [ ]:
# Validate Demand Categories

print("Total District Records:", len(district_demand))

print("\nDemand Categories:")
print(
    district_demand["Demand_Category"]
    .value_counts(dropna=False)
)

print(
    "\nMissing Retail Demand Index:",
    district_demand["Retail_Demand_Index"].isna().sum()
)

print(
    "Missing Demand Category:",
    district_demand["Demand_Category"].isna().sum()
)

Total District Records: 640

Demand Categories:
Demand_Category
Moderate     163
Low          160
Very High    159
High         158
Name: count, dtype: int64

Missing Retail Demand Index: 0
Missing Demand Category: 0


In [ ]:
# Create District Demand Ranking

district_demand["Demand_Rank"] = (
    district_demand["Retail_Demand_Index"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

district_demand[
    [
        "State",
        "District",
        "Retail_Demand_Index",
        "Demand_Rank"
    ]
].sort_values("Demand_Rank").head(10)

,State,District,Retail_Demand_Index,Demand_Rank
1713,29,Bangalore,0.899,1
1560,27,Pune,0.878,2
1551,27,Mumbai Suburban,0.877,3
1806,33,Chennai,0.864,4
1893,33,Coimbatore,0.862,5
1023,19,Kolkata,0.861,6
1512,27,Nagpur,0.860,7
1473,24,Surat,0.860,7
1545,27,Nashik,0.851,8
1563,27,Ahmadnagar,0.851,8


In [ ]:
# Save Engineered Feature Datasets

stores_features.to_csv(
    "store_competition_features.csv",
    index=False
)

district_demand.to_csv(
    "district_demand_features.csv",
    index=False
)

competition_features.to_csv(
    "location_competition_features.csv",
    index=False
)

print("Feature engineering datasets created successfully.")

print("\nStore Features:", stores_features.shape)
print("District Demand Features:", district_demand.shape)
print("Location Competition Features:", competition_features.shape)

Feature engineering datasets created successfully.

Store Features: (4081, 21)
District Demand Features: (640, 10)
Location Competition Features: (607, 8)


In [ ]:
# Normalize Retail Demand Score

min_demand = district_demand["Retail_Demand_Index"].min()
max_demand = district_demand["Retail_Demand_Index"].max()

district_demand["Demand_Score"] = (
    (district_demand["Retail_Demand_Index"] - min_demand) /
    (max_demand - min_demand)
) * 100

district_demand[
    [
        "District",
        "Retail_Demand_Index",
        "Demand_Rank",
        "Demand_Score"
    ]
].sort_values(
    "Demand_Score",
    ascending=False
).head(10).round(2)


,District,Retail_Demand_Index,Demand_Rank,Demand_Score
1713,Bangalore,0.90,1,100.00
1560,Pune,0.88,2,97.15
1551,Mumbai Suburban,0.88,3,97.01
1806,Chennai,0.86,4,95.24
1893,Coimbatore,0.86,5,94.97
1023,Kolkata,0.86,6,94.84
1512,Nagpur,0.86,7,94.70
1473,Surat,0.86,7,94.70
1554,Mumbai,0.85,8,93.48
1563,Ahmadnagar,0.85,8,93.48


In [ ]:
# Normalize Competition Score

min_comp = competition_features["Competition_Intensity"].min()
max_comp = competition_features["Competition_Intensity"].max()

competition_features["Competition_Score"] = (
    (competition_features["Competition_Intensity"] - min_comp) /
    (max_comp - min_comp)
) * 100

competition_features[
    [
        "location_lat",
        "location_lon",
        "Total_Dark_Stores",
        "Competing_Brands",
        "Competition_Intensity",
        "Competition_Score"
    ]
].sort_values(
    "Competition_Score",
    ascending=False
).head(10).round(2)

,location_lat,location_lon,Total_Dark_Stores,Competing_Brands,Competition_Intensity,Competition_Score
70,12.9,77.6,88,3,264,100.00
495,28.6,77.4,72,3,216,81.75
80,13.0,77.7,72,3,216,81.75
79,13.0,77.6,58,3,174,65.78
158,17.4,78.4,58,3,174,65.78
163,17.5,78.4,57,3,171,64.64
492,28.6,77.1,54,3,162,61.22
494,28.6,77.3,52,3,156,58.94
205,19.1,72.9,52,3,156,58.94
71,12.9,77.7,52,3,156,58.94


In [ ]:
# Save Engineered Feature Datasets

stores_features.to_csv(
    "store_competition_features.csv",
    index=False
)

district_demand.to_csv(
    "district_demand_features.csv",
    index=False
)

competition_features.to_csv(
    "location_competition_features.csv",
    index=False
)

print("Feature engineering completed successfully.")

print("\nStore Competition Features:", stores_features.shape)
print("District Demand Features:", district_demand.shape)
print("Location Competition Features:", competition_features.shape)

Feature engineering completed successfully.

Store Competition Features: (4081, 21)
District Demand Features: (640, 11)
Location Competition Features: (607, 9)


In [ ]:
# Download Engineered Datasets

from google.colab import files

files.download("store_competition_features.csv")
files.download("district_demand_features.csv")
files.download("location_competition_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Import Visualization Library

import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Analyze Brand Competitive Position

brand_position = (
    stores_features
    .groupby("brand")
    .agg(
        Avg_Brand_Share=("Brand_Share", "mean"),
        Avg_Competitor_Ratio=("Competitor_Ratio", "mean"),
        Avg_Competition_Intensity=("Competition_Intensity", "mean"),
        Total_Stores=("id", "count")
    )
    .reset_index()
)

# Brand identity colors
brand_colors = {
    "Blinkit": "#FFD000",
    "Zepto": "#7B2CBF",
    "Swiggy Instamart": "#FC8019"
}

fig = px.scatter(
    brand_position,
    x="Avg_Competitor_Ratio",
    y="Avg_Brand_Share",
    size="Total_Stores",
    color="brand",
    text="brand",

    color_discrete_map=brand_colors,

    hover_data={
        "Total_Stores": True,
        "Avg_Competition_Intensity": ":.2f",
        "Avg_Competitor_Ratio": ":.2f",
        "Avg_Brand_Share": ":.2f"
    },

    title="Competitive Position of Quick-Commerce Brands",

    labels={
        "Avg_Competitor_Ratio": "Average Competitor Pressure",
        "Avg_Brand_Share": "Average Brand Share",
        "Total_Stores": "Total Stores",
        "brand": "Brand"
    },

    size_max=70
)

fig.update_traces(
    textposition="top center",
    marker=dict(
        opacity=0.80,
        line=dict(
            width=1.5,
            color="white"
        )
    )
)

fig.update_layout(
    template="plotly_white",
    height=600,
    showlegend=False,

    title={
        "x": 0.5,
        "xanchor": "center"
    },

    xaxis=dict(
        tickformat=".0%"
    ),

    yaxis=dict(
        tickformat=".0%"
    )
)

fig.show()

In [ ]:
# Create Brand Competitive Profile

brand_profile = (
    stores_features
    .groupby("brand")
    .agg(
        Store_Density=("Total_Dark_Stores", "mean"),
        Competitor_Pressure=("Competitor_Ratio", "mean"),
        Brand_Share=("Brand_Share", "mean"),
        Competition_Intensity=("Competition_Intensity", "mean"),
        Brand_Diversity=("Competing_Brands", "mean")
    )
)

# Normalize features to 0-100
profile_normalized = (
    (brand_profile - brand_profile.min()) /
    (brand_profile.max() - brand_profile.min())
) * 100

profile_normalized = profile_normalized.fillna(0)

categories = profile_normalized.columns.tolist()

# Brand identity colors
brand_colors = {
    "Blinkit": "#FFD000",
    "Zepto": "#7B2CBF",
    "Swiggy Instamart": "#FC8019"
}

fig = go.Figure()

for brand in profile_normalized.index:

    values = profile_normalized.loc[brand].tolist()

    fig.add_trace(
        go.Scatterpolar(
            r=values + [values[0]],
            theta=categories + [categories[0]],

            fill="toself",
            name=brand,

            line=dict(
                color=brand_colors[brand],
                width=3
            ),

            fillcolor=brand_colors[brand],
            opacity=0.35,

            marker=dict(
                color=brand_colors[brand],
                size=7
            )
        )
    )

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100],
            ticksuffix="%"
        )
    ),

    title={
        "text": "Competitive DNA of Quick-Commerce Brands",
        "x": 0.5
    },

    legend=dict(
        orientation="h",
        y=1.10,
        x=0.5,
        xanchor="center"
    ),

    template="plotly_white",
    height=650
)

fig.show()

In [ ]:
import plotly.express as px

# ==========================================
# COMPETITION BATTLEGROUND MATRIX
# ==========================================

battle = competition_features.copy()

# Remove invalid rows
battle = battle.dropna(
    subset=[
        "Total_Dark_Stores",
        "Competing_Brands",
        "Competition_Intensity"
    ]
)

# Midpoints for quadrant separation
density_mid = battle["Total_Dark_Stores"].median()
competition_mid = battle["Competition_Intensity"].median()

# ==========================================
# FIGURE
# ==========================================

fig = px.scatter(

    battle,

    x="Total_Dark_Stores",
    y="Competition_Intensity",

    size="Competing_Brands",
    color="Competing_Brands",

    hover_data={
        "location_lat": ":.1f",
        "location_lon": ":.1f",
        "Total_Dark_Stores": True,
        "Competing_Brands": True,
        "Competition_Intensity": True
    },

    color_continuous_scale=[
        "#B8E0D2",
        "#4EA8A8",
        "#FC8019",
        "#7B2CBF"
    ],

    size_max=35
)

# ==========================================
# QUADRANT LINES
# ==========================================

fig.add_vline(
    x=density_mid,
    line_dash="dash",
    line_color="#AAAAAA"
)

fig.add_hline(
    y=competition_mid,
    line_dash="dash",
    line_color="#AAAAAA"
)

# ==========================================
# QUADRANT LABELS
# ==========================================

fig.add_annotation(
    x=0.97,
    y=0.96,
    xref="paper",
    yref="paper",
    text="<b>COMPETITIVE HOTSPOTS</b><br>Dense + Highly Contested",
    showarrow=False,
    bgcolor="rgba(123,44,191,0.10)",
    bordercolor="#7B2CBF",
    borderpad=7
)

fig.add_annotation(
    x=0.03,
    y=0.96,
    xref="paper",
    yref="paper",
    text="<b>RIVALRY POCKETS</b><br>Lower Density + High Competition",
    showarrow=False,
    bgcolor="rgba(252,128,25,0.10)",
    bordercolor="#FC8019",
    borderpad=7
)

fig.add_annotation(
    x=0.97,
    y=0.04,
    xref="paper",
    yref="paper",
    text="<b>CONCENTRATED MARKETS</b><br>Dense + Lower Rivalry",
    showarrow=False,
    bgcolor="rgba(78,168,168,0.10)",
    bordercolor="#4EA8A8",
    borderpad=7
)

fig.add_annotation(
    x=0.03,
    y=0.04,
    xref="paper",
    yref="paper",
    text="<b>LOW-PRESSURE ZONES</b><br>Sparse + Lower Competition",
    showarrow=False,
    bgcolor="rgba(184,224,210,0.18)",
    bordercolor="#79B8A5",
    borderpad=7
)

# ==========================================
# DESIGN
# ==========================================

fig.update_traces(
    marker=dict(
        opacity=0.72,
        line=dict(
            color="white",
            width=1.2
        )
    )
)

fig.update_layout(

    title=dict(
        text=(
            "<b>WHERE ARE INDIA'S QUICK-COMMERCE BATTLEGROUNDS?</b><br>"
            "<span style='font-size:13px;color:#777777'>"
            "Separating concentrated markets from competitive saturation"
            "</span>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(
            size=22,
            color="#263B5E"
        )
    ),

    xaxis=dict(
        title="Dark-Store Density →",
        gridcolor="#EEEEEE",
        zeroline=False
    ),

    yaxis=dict(
        title="Competition Intensity →",
        gridcolor="#EEEEEE",
        zeroline=False
    ),

    coloraxis_colorbar=dict(
        title="Brands<br>Present"
    ),

    plot_bgcolor="#FCFCFC",
    paper_bgcolor="white",

    width=1000,
    height=650,

    margin=dict(
        t=125,
        l=90,
        r=120,
        b=80
    )
)

fig.show()

In [ ]:
# Compare Geographic Footprint by Brand

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

brands = {
    "Blinkit": {
        "column": "blinkit",
        "color": "#FFD000"
    },
    "Zepto": {
        "column": "zepto",
        "color": "#7B2CBF"
    },
    "Swiggy Instamart": {
        "column": "swiggy_instamart",
        "color": "#FC8019"
    }
}

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Blinkit",
        "Zepto",
        "Swiggy Instamart"
    ],
    specs=[[
        {"type": "map"},
        {"type": "map"},
        {"type": "map"}
    ]]
)

for i, (brand, info) in enumerate(brands.items(), start=1):

    data = competition_features[
        competition_features[info["column"]] > 0
    ].copy()

    marker_size = (
        3 + np.sqrt(data[info["column"]]) * 2
    )

    fig.add_trace(
        go.Scattermap(
            lat=data["location_lat"],
            lon=data["location_lon"],

            mode="markers",

            marker=dict(
                size=marker_size,
                color=info["color"],
                opacity=0.7
            ),

            customdata=data[
                [
                    info["column"],
                    "Total_Dark_Stores",
                    "Competition_Intensity"
                ]
            ].values,

            hovertemplate=(
                f"<b>{brand}</b><br>"
                "Brand Stores: %{customdata[0]}<br>"
                "Total Cluster Stores: %{customdata[1]}<br>"
                "Competition Intensity: %{customdata[2]}"
                "<extra></extra>"
            ),

            showlegend=False
        ),

        row=1,
        col=i
    )

# Same India view for all three maps

for map_name in ["map", "map2", "map3"]:

    fig.layout[map_name].update(
        style="carto-positron",

        center=dict(
            lat=22.5,
            lon=79
        ),

        zoom=2.7
    )

fig.update_layout(
    title={
        "text": "Geographic Footprint Comparison of Quick-Commerce Brands",
        "x": 0.5
    },

    height=600,
    width=1400,

    margin=dict(
        l=10,
        r=10,
        t=90,
        b=10
    )
)

fig.show()

In [ ]:
# Compare Brand Share Across Major Locations

top_locations = (
    competition_features
    .nlargest(15, "Total_Dark_Stores")
    .copy()
)

top_locations["Location"] = (
    top_locations["location_lat"].round(1).astype(str)
    + ", "
    + top_locations["location_lon"].round(1).astype(str)
)

# Calculate percentage share
top_locations["Blinkit_Share"] = (
    top_locations["blinkit"] /
    top_locations["Total_Dark_Stores"] * 100
)

top_locations["Zepto_Share"] = (
    top_locations["zepto"] /
    top_locations["Total_Dark_Stores"] * 100
)

top_locations["Instamart_Share"] = (
    top_locations["swiggy_instamart"] /
    top_locations["Total_Dark_Stores"] * 100
)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        y=top_locations["Location"],
        x=top_locations["Blinkit_Share"],
        name="Blinkit",
        orientation="h",
        marker_color="#FFD000",
        customdata=top_locations["blinkit"],
        hovertemplate=(
            "<b>Blinkit</b><br>"
            "Share: %{x:.1f}%<br>"
            "Stores: %{customdata}"
            "<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Bar(
        y=top_locations["Location"],
        x=top_locations["Zepto_Share"],
        name="Zepto",
        orientation="h",
        marker_color="#7B2CBF",
        customdata=top_locations["zepto"],
        hovertemplate=(
            "<b>Zepto</b><br>"
            "Share: %{x:.1f}%<br>"
            "Stores: %{customdata}"
            "<extra></extra>"
        )
    )
)

fig.add_trace(
    go.Bar(
        y=top_locations["Location"],
        x=top_locations["Instamart_Share"],
        name="Swiggy Instamart",
        orientation="h",
        marker_color="#FC8019",
        customdata=top_locations["swiggy_instamart"],
        hovertemplate=(
            "<b>Swiggy Instamart</b><br>"
            "Share: %{x:.1f}%<br>"
            "Stores: %{customdata}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title="Brand Market Share Across Major Quick-Commerce Locations",

    barmode="stack",

    xaxis=dict(
        title="Share of Dark Stores (%)",
        range=[0, 100]
    ),

    yaxis=dict(
        title="Location Cluster",
        autorange="reversed"
    ),

    legend=dict(
        title="Brand",
        orientation="h",
        y=1.08,
        x=0.5,
        xanchor="center"
    ),

    template="plotly_white",
    height=700
)

fig.show()